In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
# ============================================================
# PRODUCTION EEGNET-LARGE ARCHITECTURE (DYNAMIC SHAPE FIXED)
# ============================================================

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score

# ============================================================
# 1. LOAD DATASETS
# ============================================================
X_train = np.load("/kaggle/input/datasets/alekhya7gangopadhyay/eeg-processed/X_train_500.npy")
X_test  = np.load("/kaggle/input/datasets/alekhya7gangopadhyay/eeg-processed/X_test_500.npy")
y_train = np.load("/kaggle/input/datasets/alekhya7gangopadhyay/eeg-processed/y_train_cls_500.npy")
y_test  = np.load("/kaggle/input/datasets/alekhya7gangopadhyay/eeg-processed/y_test_cls_500.npy")

print(f"Dataset arrays loaded successfully.")
print(f"Train Inputs: {X_train.shape} -> Test Inputs: {X_test.shape}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training EEGNet-Large on computation engine:", device)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
y_test_t  = torch.tensor(y_test, dtype=torch.long)

# ============================================================
# 2. DATA LOADERS SETUP
# ============================================================

BATCH_SIZE = 256
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE, shuffle=False)

# ============================================================
# 3. FIXED EEGNET-LARGE ARCHITECTURE DEFINITION
# ============================================================

class EEGNet_Large(nn.Module):
    def __init__(self, num_channels=3, num_classes=4, F1=16, D=4, F2=64, chunk_size=255):
        super().__init__()
        
        # --- BLOCK 1: Temporal & Depthwise Spatial Convolutions ---
        self.temporal_conv = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(F1)
        )
        
        self.depthwise_conv = nn.Sequential(
            nn.Conv2d(F1, F1 * D, kernel_size=(num_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(), 
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(0.3)
        )
        
        # --- BLOCK 2: Separable Convolutions ---
        self.separable_conv = nn.Sequential(
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F2, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(0.4)
        )
        
        # FIX: Dynamically calculate flat_features_dim using a mock tensor pass
        self.flat_features_dim = self._get_flat_dim(num_channels, chunk_size)
        print(f"-> Successfully calculated dynamic flat embedding size: {self.flat_features_dim} features.")
        
        # --- BLOCK 3: Classification Dense Layer ---
        self.classifier_head = nn.Sequential(
            nn.Linear(self.flat_features_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes)
        )

    def _get_flat_dim(self, num_channels, chunk_size):
        """Helper method to run a dummy forward pass and catch the shape size"""
        with torch.no_grad():
            dummy_input = torch.zeros(1, 1, num_channels, chunk_size)
            x = self.temporal_conv(dummy_input)
            x = self.depthwise_conv(x)
            x = self.separable_conv(x)
            return x.view(1, -1).size(1)

    def forward(self, x):
        # Format change: (batch, time, channels) -> (batch, 1, channels, time)
        x = x.transpose(1, 2).unsqueeze(1)
        
        x = self.temporal_conv(x)
        x = self.depthwise_conv(x)
        x = self.separable_conv(x)
        
        # Flatten dynamically to vector dimension
        x = x.view(x.size(0), -1)
        logits = self.classifier_head(x)
        return logits

# ============================================================
# 4. OPTIMIZATION & SCHEDULER SETUP
# ============================================================

model = EEGNet_Large(num_channels=3, num_classes=4).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)

# ============================================================
# 5. CORE OPTIMIZATION TRAINING LOOP
# ============================================================

EPOCHS = 20
train_losses, test_losses = [], []
train_accs, test_accs = [], []
best_test_acc = 0.0

print("\nStarting Corrected EEGNet-Large Model Training...")
for epoch in range(EPOCHS):
    
    # --- Training State ---
    model.train()
    total_train_loss = 0
    correct_train = 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        logits = model(X_batch)
        
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item() * X_batch.size(0)
        preds = torch.argmax(logits, dim=1)
        correct_train += (preds == y_batch).sum().item()
        
    avg_train_loss = total_train_loss / len(train_loader.dataset)
    epoch_train_acc = (correct_train / len(train_loader.dataset)) * 100
    
    # --- Evaluation State ---
    model.eval()
    total_test_loss = 0
    correct_test = 0
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            
            total_test_loss += loss.item() * X_batch.size(0)
            preds = torch.argmax(logits, dim=1)
            correct_test += (preds == y_batch).sum().item()
            
    avg_test_loss = total_test_loss / len(test_loader.dataset)
    epoch_test_acc = (correct_test / len(test_loader.dataset)) * 100
    
    scheduler.step(epoch_test_acc)
    
    train_losses.append(avg_train_loss)
    test_losses.append(avg_test_loss)
    train_accs.append(epoch_train_acc)
    test_accs.append(epoch_test_acc)
    
    if epoch_test_acc > best_test_acc:
        best_test_acc = epoch_test_acc
        torch.save(model.state_dict(), "/kaggle/working/EEGNet_Large_classifier_updated.pth")
        
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
          f"| Train Loss: {avg_train_loss:.4f} ({epoch_train_acc:.2f}% Acc) "
          f"| Test Loss: {avg_test_loss:.4f} ({epoch_test_acc:.2f}% Test Acc)")

print(f"\nTraining Complete. Best EEGNet-Large Test Accuracy: {best_test_acc:.2f}%")
print("Saved production weights to: /kaggle/working/EEGNet_Large_classifier.pth")

Dataset arrays loaded successfully.
Train Inputs: (63544, 255, 3) -> Test Inputs: (15840, 255, 3)
Training EEGNet-Large on computation engine: cuda
-> Successfully calculated dynamic flat embedding size: 512 features.

Starting Corrected EEGNet-Large Model Training...
Epoch [01/20] | Train Loss: 1.3287 (37.37% Acc) | Test Loss: 1.1942 (47.17% Test Acc)
Epoch [02/20] | Train Loss: 1.2034 (46.42% Acc) | Test Loss: 1.1007 (53.24% Test Acc)
Epoch [03/20] | Train Loss: 1.1382 (50.59% Acc) | Test Loss: 1.0184 (56.77% Test Acc)
Epoch [04/20] | Train Loss: 1.0840 (53.39% Acc) | Test Loss: 0.9467 (61.22% Test Acc)
Epoch [05/20] | Train Loss: 1.0378 (56.09% Acc) | Test Loss: 0.8856 (64.76% Test Acc)
Epoch [06/20] | Train Loss: 1.0005 (58.28% Acc) | Test Loss: 0.8382 (67.58% Test Acc)
Epoch [07/20] | Train Loss: 0.9677 (60.24% Acc) | Test Loss: 0.7871 (70.04% Test Acc)
Epoch [08/20] | Train Loss: 0.9456 (61.19% Acc) | Test Loss: 0.7488 (71.58% Test Acc)
Epoch [09/20] | Train Loss: 0.9284 (61.95% 

In [2]:
# ============================================================
# STANDALONE TESTING WORKBOOK: FIXED EEGNET-LARGE PREDICTOR
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import Counter

# ============================================================
# 1. FIXED ARCHITECTURE DEFINITION (DYNAMIC SHAPE ALIGNED)
# ============================================================

class EEGNet_Large(nn.Module):
    def __init__(self, num_channels=3, num_classes=4, F1=16, D=4, F2=64, chunk_size=255):
        super().__init__()
        
        self.temporal_conv = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(F1)
        )
        
        self.depthwise_conv = nn.Sequential(
            nn.Conv2d(F1, F1 * D, kernel_size=(num_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(0.3)
        )
        
        self.separable_conv = nn.Sequential(
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F2, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(0.4)
        )
        
        # FIX: Dynamically calculate flat_features_dim via a mock tensor pass to ensure it matches 512
        self.flat_features_dim = self._get_flat_dim(num_channels, chunk_size)
        
        self.classifier_head = nn.Sequential(
            nn.Linear(self.flat_features_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes)
        )

    def _get_flat_dim(self, num_channels, chunk_size):
        """Helper method to determine exact output flattening dimensions"""
        with torch.no_grad():
            dummy_input = torch.zeros(1, 1, num_channels, chunk_size)
            x = self.temporal_conv(dummy_input)
            x = self.depthwise_conv(x)
            x = self.separable_conv(x)
            return x.view(1, -1).size(1)

    def forward(self, x):
        # Format change: (batch, time, channels) -> (batch, 1, channels, time)
        x = x.transpose(1, 2).unsqueeze(1)
        x = self.temporal_conv(x)
        x = self.depthwise_conv(x)
        x = self.separable_conv(x)
        x = x.view(x.size(0), -1)
        return self.classifier_head(x)

# ============================================================
# 2. CONFIGURATIONS & ALIGNMENT
# ============================================================

NEW_UPLOADED_EXCEL = "/kaggle/input/datasets/alekhya7gangopadhyay/for-test/For.xlsx"
MODEL_WEIGHTS_PATH = "/kaggle/input/models/alekhya7gangopadhyay/eeg-net/pytorch/default/1/EEGNet_Large_classifier_updated.pth" # Verifying path filename matches training output

SELECTED_CHANNELS = ["P4 - O2", "P3 - O1", "F4 - C4"]
EXPECTED_TIMESTEPS = 256

# Aligned Direction Map
DIRECTION_MAP = {
   0: "RIGHT", 1: "LEFT", 2: "FORWARD", 3: "BACKWARD"
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# 3. LOAD, STRIP & STANDARDIZE EXCEL ROWS
# ============================================================

try:
    df = pd.read_excel(NEW_UPLOADED_EXCEL)
    df.columns = df.columns.str.strip()
    df_selected = df[SELECTED_CHANNELS]
    raw_values = df_selected.values.astype(np.float32)
    
    if raw_values.shape[0] >= EXPECTED_TIMESTEPS:
        num_windows = raw_values.shape[0] // EXPECTED_TIMESTEPS
        X_eval = raw_values[:num_windows * EXPECTED_TIMESTEPS].reshape(num_windows, EXPECTED_TIMESTEPS, 3)
        
        # Z-score standardization normalization block
        X_eval_2d = X_eval.reshape(-1, 3)
        mean, std = np.mean(X_eval_2d, axis=0), np.std(X_eval_2d, axis=0)
        std = np.where(std == 0, 1e-8, std)
        
        X_eval = ((X_eval_2d - mean) / std).reshape(num_windows, EXPECTED_TIMESTEPS, 3)
        X_tensor = torch.tensor(X_eval, dtype=torch.float32).to(device)
    else:
        raise ValueError("Excel row dimensions too small.")
except Exception as e:
    print(f"Pipeline processing failure: {e}"); exit()

# ============================================================
# 4. INITIALIZE MODEL & EVALUATE MATRIX
# ============================================================

model = EEGNet_Large(num_channels=3, num_classes=4).to(device)
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
model.eval()

with torch.no_grad():
    logits = model(X_tensor)
    probabilities = torch.softmax(logits, dim=1)
    predicted_indices = torch.argmax(probabilities, dim=1).cpu().numpy()
    confidence_scores = probabilities.cpu().numpy()

# ============================================================
# 5. OUTPUT DISPATCH
# ============================================================

print("\n==================================================")
print("          EEGNET-LARGE WINDOW SEGMENTATION        ")
print("==================================================")
for i, pred_idx in enumerate(predicted_indices):
    print(f"Block #{i+1:02d} | Path: ➔ 【 {DIRECTION_MAP[pred_idx]} 】 (Confidence: {confidence_scores[i, pred_idx]*100:.2f}%)")

prediction_counts = Counter(predicted_indices)
final_winning_idx = prediction_counts.most_common(1)[0][0]
avg_confidence = np.mean([confidence_scores[i, final_winning_idx] for i, p in enumerate(predicted_indices) if p == final_winning_idx]) * 100

print("\n==================================================")
print("              FINAL DECODED OUTBOUND              ")
print("==================================================")
print(f"★ OVERARCHING PREDICTION : ➔ 【 {DIRECTION_MAP[final_winning_idx]} 】")
print(f"★ CONSENSUS CONFIDENCE   : {avg_confidence:.2f}%")
print("==================================================")


          EEGNET-LARGE WINDOW SEGMENTATION        
Block #01 | Path: ➔ 【 BACKWARD 】 (Confidence: 44.04%)
Block #02 | Path: ➔ 【 FORWARD 】 (Confidence: 41.05%)
Block #03 | Path: ➔ 【 FORWARD 】 (Confidence: 77.77%)
Block #04 | Path: ➔ 【 FORWARD 】 (Confidence: 46.36%)
Block #05 | Path: ➔ 【 BACKWARD 】 (Confidence: 66.43%)
Block #06 | Path: ➔ 【 FORWARD 】 (Confidence: 35.11%)
Block #07 | Path: ➔ 【 FORWARD 】 (Confidence: 29.11%)
Block #08 | Path: ➔ 【 FORWARD 】 (Confidence: 35.58%)
Block #09 | Path: ➔ 【 FORWARD 】 (Confidence: 35.87%)

              FINAL DECODED OUTBOUND              
★ OVERARCHING PREDICTION : ➔ 【 FORWARD 】
★ CONSENSUS CONFIDENCE   : 42.98%


In [3]:
# ============================================================
# STANDALONE TESTING WORKBOOK: FIXED EEGNET-LARGE PREDICTOR
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import Counter

# ============================================================
# 1. FIXED ARCHITECTURE DEFINITION (DYNAMIC SHAPE ALIGNED)
# ============================================================

class EEGNet_Large(nn.Module):
    def __init__(self, num_channels=3, num_classes=4, F1=16, D=4, F2=64, chunk_size=255):
        super().__init__()
        
        self.temporal_conv = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(F1)
        )
        
        self.depthwise_conv = nn.Sequential(
            nn.Conv2d(F1, F1 * D, kernel_size=(num_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(0.3)
        )
        
        self.separable_conv = nn.Sequential(
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F2, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(0.4)
        )
        
        # FIX: Dynamically calculate flat_features_dim via a mock tensor pass to ensure it matches 512
        self.flat_features_dim = self._get_flat_dim(num_channels, chunk_size)
        
        self.classifier_head = nn.Sequential(
            nn.Linear(self.flat_features_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes)
        )

    def _get_flat_dim(self, num_channels, chunk_size):
        """Helper method to determine exact output flattening dimensions"""
        with torch.no_grad():
            dummy_input = torch.zeros(1, 1, num_channels, chunk_size)
            x = self.temporal_conv(dummy_input)
            x = self.depthwise_conv(x)
            x = self.separable_conv(x)
            return x.view(1, -1).size(1)

    def forward(self, x):
        # Format change: (batch, time, channels) -> (batch, 1, channels, time)
        x = x.transpose(1, 2).unsqueeze(1)
        x = self.temporal_conv(x)
        x = self.depthwise_conv(x)
        x = self.separable_conv(x)
        x = x.view(x.size(0), -1)
        return self.classifier_head(x)

# ============================================================
# 2. CONFIGURATIONS & ALIGNMENT
# ============================================================

NEW_UPLOADED_EXCEL = "/kaggle/input/datasets/alekhya7gangopadhyay/test-ry/RY.xlsx"
MODEL_WEIGHTS_PATH = "/kaggle/input/models/alekhya7gangopadhyay/eeg-net/pytorch/default/1/EEGNet_Large_classifier_updated.pth" # Verifying path filename matches training output

SELECTED_CHANNELS = ["P4 - O2", "P3 - O1", "F4 - C4"]
EXPECTED_TIMESTEPS = 256

# Aligned Direction Map
DIRECTION_MAP = {
   0: "RIGHT", 1: "LEFT", 2: "FORWARD", 3: "BACKWARD"
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# 3. LOAD, STRIP & STANDARDIZE EXCEL ROWS
# ============================================================

try:
    df = pd.read_excel(NEW_UPLOADED_EXCEL)
    df.columns = df.columns.str.strip()
    df_selected = df[SELECTED_CHANNELS]
    raw_values = df_selected.values.astype(np.float32)
    
    if raw_values.shape[0] >= EXPECTED_TIMESTEPS:
        num_windows = raw_values.shape[0] // EXPECTED_TIMESTEPS
        X_eval = raw_values[:num_windows * EXPECTED_TIMESTEPS].reshape(num_windows, EXPECTED_TIMESTEPS, 3)
        
        # Z-score standardization normalization block
        X_eval_2d = X_eval.reshape(-1, 3)
        mean, std = np.mean(X_eval_2d, axis=0), np.std(X_eval_2d, axis=0)
        std = np.where(std == 0, 1e-8, std)
        
        X_eval = ((X_eval_2d - mean) / std).reshape(num_windows, EXPECTED_TIMESTEPS, 3)
        X_tensor = torch.tensor(X_eval, dtype=torch.float32).to(device)
    else:
        raise ValueError("Excel row dimensions too small.")
except Exception as e:
    print(f"Pipeline processing failure: {e}"); exit()

# ============================================================
# 4. INITIALIZE MODEL & EVALUATE MATRIX
# ============================================================

model = EEGNet_Large(num_channels=3, num_classes=4).to(device)
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
model.eval()

with torch.no_grad():
    logits = model(X_tensor)
    probabilities = torch.softmax(logits, dim=1)
    predicted_indices = torch.argmax(probabilities, dim=1).cpu().numpy()
    confidence_scores = probabilities.cpu().numpy()

# ============================================================
# 5. OUTPUT DISPATCH
# ============================================================

print("\n==================================================")
print("          EEGNET-LARGE WINDOW SEGMENTATION        ")
print("==================================================")
for i, pred_idx in enumerate(predicted_indices):
    print(f"Block #{i+1:02d} | Path: ➔ 【 {DIRECTION_MAP[pred_idx]} 】 (Confidence: {confidence_scores[i, pred_idx]*100:.2f}%)")

prediction_counts = Counter(predicted_indices)
final_winning_idx = prediction_counts.most_common(1)[0][0]
avg_confidence = np.mean([confidence_scores[i, final_winning_idx] for i, p in enumerate(predicted_indices) if p == final_winning_idx]) * 100

print("\n==================================================")
print("              FINAL DECODED OUTBOUND              ")
print("==================================================")
print(f"★ OVERARCHING PREDICTION : ➔ 【 {DIRECTION_MAP[final_winning_idx]} 】")
print(f"★ CONSENSUS CONFIDENCE   : {avg_confidence:.2f}%")
print("==================================================")


          EEGNET-LARGE WINDOW SEGMENTATION        
Block #01 | Path: ➔ 【 RIGHT 】 (Confidence: 35.78%)
Block #02 | Path: ➔ 【 RIGHT 】 (Confidence: 41.45%)
Block #03 | Path: ➔ 【 FORWARD 】 (Confidence: 66.74%)
Block #04 | Path: ➔ 【 BACKWARD 】 (Confidence: 40.20%)
Block #05 | Path: ➔ 【 FORWARD 】 (Confidence: 37.08%)
Block #06 | Path: ➔ 【 BACKWARD 】 (Confidence: 41.73%)
Block #07 | Path: ➔ 【 RIGHT 】 (Confidence: 35.25%)
Block #08 | Path: ➔ 【 FORWARD 】 (Confidence: 58.46%)
Block #09 | Path: ➔ 【 FORWARD 】 (Confidence: 45.25%)

              FINAL DECODED OUTBOUND              
★ OVERARCHING PREDICTION : ➔ 【 FORWARD 】
★ CONSENSUS CONFIDENCE   : 51.88%
